# 1. PROJECT CONTEXT

## Research Title
**"Analisis Performa Algoritma Machine Learning untuk Klasifikasi Jenis dan Tingkat Keparahan Cyberbullying pada Teks Bahasa Indonesia Menggunakan TF-IDF"**
*(Performance Analysis of Machine Learning Algorithms for Cyberbullying Type and Severity Classification in Indonesian Text Using TF-IDF)*

## Role of Preprocessing
This is the sixth stage of the pipeline. The goal is to transform the raw text into a clean, standardized format suitable for the upcoming TF-IDF feature extraction (`06_tfidf.ipynb`).

## Why Preprocessing Occurs After Validation
Validation (`04_validation.ipynb`) ensures we have a structurally sound and correctly labeled dataset. Preprocessing assumes a valid schema and focuses entirely on text transformation.

## Preservation of Original Data
The original text must remain completely intact. We will create a new column, `text_processed`, for the cleaned version. This ensures maximum traceability, allowing error analysis and explainability stages to reference the exact words a user typed.

## Reproducibility
The preprocessing rules are fully deterministic and apply identically to every text without data leakage (no model predictions or test-set statistics are used).


# 2. PREPROCESSING OBJECTIVES

Our goals for this stage are to:
- **Improve Text Consistency**: Reduce vocabulary variation (e.g., lowercasing).
- **Reduce Irrelevant Noise**: Strip URLs, HTML tags, and non-alphanumeric punctuation.
- **Preserve Semantic Information**: Intentionally keep negations, slang, and emotional indicators that define cyberbullying context.
- **Prepare for TF-IDF**: Ensure the output consists of clean, space-separated tokens.
- **Avoid Data Leakage**: Do not use labels or future model knowledge to transform text.
- **Preserve Auditing Data**: Never overwrite the original text.


In [1]:
# 3. IMPORT LIBRARIES

import pandas as pd
import numpy as np
import pathlib
import re
import unicodedata
import warnings

warnings.filterwarnings('ignore')
print("Libraries imported successfully.")


Libraries imported successfully.


In [2]:
# 4. CONFIGURATION

PROJECT_ROOT = pathlib.Path.cwd().parent
DATA_VALIDATED_DIR = PROJECT_ROOT / "data" / "validated"
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RAW_DIR = PROJECT_ROOT / "data" / "raw"\nBASE_REPORTS_DIR = PROJECT_ROOT / "reports"\nREPORTS_DIR = BASE_REPORTS_DIR / "preprocessing"

VALIDATED_DATA_PATH = DATA_VALIDATED_DIR / "validated_cyberbullying_dataset.csv"
PROCESSED_DATA_PATH = DATA_PROCESSED_DIR / "processed_cyberbullying_dataset.csv"
PREPROCESSING_SUMMARY_PATH = REPORTS_DIR / "preprocessing_summary.csv"

# Columns
TEXT_COL = "text"
TEXT_PROCESSED_COL = "text_processed"
LABEL_COL = "cyberbullying_type"
PROVENANCE_COLS = ["source_dataset", "original_row_id", "severity_level", "original_cyberbullying_type", "original_severity_level"]

DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Validated Data Path: {VALIDATED_DATA_PATH}")


Validated Data Path: /home/zapp/Kampus/PM-NEW/data/validated/validated_cyberbullying_dataset.csv


In [3]:
# 5. LOAD VALIDATED DATASET

if not VALIDATED_DATA_PATH.exists():
    raise FileNotFoundError(f"FAIL: Validated dataset not found at {VALIDATED_DATA_PATH}. Please run 04_validation.ipynb first.")

df = pd.read_csv(VALIDATED_DATA_PATH)
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Columns: {list(df.columns)}")
display(df.head())


Dataset Shape: 42954 rows, 8 columns
Columns: ['text', 'cyberbullying_type', 'severity_level', 'original_cyberbullying_type', 'original_severity_level', 'source_dataset', 'source_file', 'original_row_id']


,text,cyberbullying_type,severity_level,original_cyberbullying_type,original_severity_level,source_dataset,source_file,original_row_id
0,setiap orang adalah seorang gadis yang akan me...,hate_speech,NaN,age,NaN,cyberbullying_cleaned_indo,cyberbullying_cleaned_indo.csv,0
1,bahwa pos ab kpop stans pergi ke sekolah bersa...,hate_speech,NaN,age,NaN,cyberbullying_cleaned_indo,cyberbullying_cleaned_indo.csv,1
2,karena beberapa orang tidak ada yang lebih bai...,hate_speech,NaN,age,NaN,cyberbullying_cleaned_indo,cyberbullying_cleaned_indo.csv,2
3,bro aku pelatih jv tahun lalu di skyline dan a...,hate_speech,NaN,age,NaN,cyberbullying_cleaned_indo,cyberbullying_cleaned_indo.csv,3
4,wanitawanita ini benarbenar mengingatkan saya ...,hate_speech,NaN,age,NaN,cyberbullying_cleaned_indo,cyberbullying_cleaned_indo.csv,4


# 5. LOAD DOMAIN LEXICONS\nLoading specific dictionaries to enrich the machine learning context during preprocessing.\n

In [ ]:
ABUSIVE_PATH = RAW_DIR / 'abusive.csv'\nHARASSMENT_PATH = RAW_DIR / 'harassment.csv'\nINSULT_PATH = RAW_DIR / 'insult.csv'\nTHREAT_PATH = RAW_DIR / 'threat.csv'\n\ndef load_lexicon(path):\n    if path.exists():\n        df = pd.read_csv(path)\n        col = df.columns[0]\n        return set(df[col].dropna().astype(str).str.lower().str.strip())\n    return set()\n\nabusive_lex = load_lexicon(ABUSIVE_PATH)\nharassment_lex = load_lexicon(HARASSMENT_PATH)\ninsult_lex = load_lexicon(INSULT_PATH)\nthreat_lex = load_lexicon(THREAT_PATH)\n\nprint(f\"Loaded {len(abusive_lex)} abusive words, {len(harassment_lex)} harassment words, {len(insult_lex)} insult words, {len(threat_lex)} threat words.\")\n

In [4]:
# 6. VERIFY INPUT DATASET

required_cols = [TEXT_COL, LABEL_COL]
missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    raise ValueError(f"CRITICAL ERROR: Dataset is missing required columns: {missing_cols}")

if df.empty:
    raise ValueError("CRITICAL ERROR: Dataset is empty.")

print("Input dataset verified. Required columns are present.")


Input dataset verified. Required columns are present.


In [5]:
# 7. INSPECT TEXT DATA

# Calculate baseline statistics
df['char_count_orig'] = df[TEXT_COL].astype(str).apply(len)
df['word_count_orig'] = df[TEXT_COL].astype(str).apply(lambda x: len(x.split()))

print("### Original Text Statistics ###")
print(f"Total Rows: {len(df)}")
print(f"Missing Values: {df[TEXT_COL].isnull().sum()}")
print(f"Empty Strings: {(df[TEXT_COL].astype(str).str.strip() == '').sum()}")
print(f"Mean Char Count: {df['char_count_orig'].mean():.2f}")
print(f"Mean Word Count: {df['word_count_orig'].mean():.2f}")
print(f"Min Word Count: {df['word_count_orig'].min()}")
print(f"Max Word Count: {df['word_count_orig'].max()}")


### Original Text Statistics ###
Total Rows: 42954
Missing Values: 0
Empty Strings: 0
Mean Char Count: 241.80
Mean Word Count: 34.64
Min Word Count: 1
Max Word Count: 880


# 8. DEFINE PREPROCESSING STRATEGY

To maximize the performance of a classical TF-IDF model on Indonesian social media cyberbullying text, we must balance noise reduction with context preservation.

| Step | Applied? | Reason |
|---|---|---|
| **Lowercasing** | YES | Eliminates case-based vocabulary duplication (e.g., "Bodoh" vs "bodoh"). |
| **URL Removal** | YES | Links provide no inherent NLP semantic value for cyberbullying. |
| **HTML Removal** | YES | Strips web scraping artifacts (e.g., `<br>`, `&amp;`). |
| **Username Mention Handling**| YES | Replaces `@username` with `USER_MENTION` to prevent the model from memorizing specific victims/aggressors. |
| **Hashtag Handling** | MODIFIED | Removes the `#` symbol but keeps the word itself (e.g., `#idiot` $\rightarrow$ `idiot`) to preserve insult context. |
| **Punctuation Removal** | YES | Removes non-alphanumeric characters. TF-IDF does not inherently capture sequential punctuation meaning. |
| **Whitespace Normalization** | YES | Removes extra spaces, tabs, and newlines. |
| **Repeated Character Normalization** | NO | Repeated chars (e.g., `anjinggggg`) denote high emotion and severity. TF-IDF will capture these as distinct high-severity tokens. |
| **Slang Normalization** | NO | We lack an authoritative, project-specific slang dictionary. Blind normalization risks destroying context. |
| **Stopword Removal** | NO | Words like "tidak", "bukan", and "jangan" are critical for negation. Removing them destroys context (e.g., "saya tidak bodoh" $\rightarrow$ "saya bodoh"). |
| **Stemming/Lemmatization** | NO | Indonesian stemmers (like Sastrawi) often fail on internet slang and can accidentally neutralize aggressive terms. |


In [6]:
# 9. DEFINE PREPROCESSING FUNCTION

def preprocess_text(text):
    if pd.isna(text):
        return ""
        
    # Ensure string
    text = str(text)
    
    # 1. Lowercase
    text = text.lower()
    
    # 2. URL Removal
    text = re.sub(r'http\S+|www\.\S+', '', text)
    
    # 3. HTML Tag Removal
    text = re.sub(r'<.*?>', '', text)
    
    # 4. Username Mention Replacement
    text = re.sub(r'@\w+', 'USER_MENTION', text)
    
    # 5. Hashtag Handling (Remove #, keep word)
    text = re.sub(r'#(\w+)', r'', text)
    
    # 6. Punctuation Removal (Keep alphanumeric and spaces)
    text = re.sub(r'[^\w\s]', ' ', text)
    
    # 7. Whitespace Normalization
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

print("Preprocessing function defined deterministically.")


Preprocessing function defined deterministically.


In [7]:
# 10. APPLY PREPROCESSING

print("Applying preprocessing. This may take a moment depending on dataset size...")

# The original text remains untouched in TEXT_COL. 
# We map the results directly to TEXT_PROCESSED_COL.
df[TEXT_PROCESSED_COL] = df[TEXT_COL].apply(preprocess_text)

print("Preprocessing complete.")


Applying preprocessing. This may take a moment depending on dataset size...
Preprocessing complete.


In [8]:
# 11. BEFORE-AND-AFTER COMPARISON

# Calculate new stats
df['char_count_proc'] = df[TEXT_PROCESSED_COL].apply(len)
df['word_count_proc'] = df[TEXT_PROCESSED_COL].apply(lambda x: len(x.split()))

df['char_reduction'] = df['char_count_orig'] - df['char_count_proc']
df['word_reduction'] = df['word_count_orig'] - df['word_count_proc']

print("### Examples of Processed Text ###")
sample_df = df.sample(n=10, random_state=42)[[TEXT_COL, TEXT_PROCESSED_COL]]
pd.set_option('display.max_colwidth', 150)
display(sample_df)

print(f"Average Character Reduction: {df['char_reduction'].mean():.2f} chars/row")
print(f"Average Word Reduction: {df['word_reduction'].mean():.2f} words/row")


### Examples of Processed Text ###


,text,text_processed
16662,yuk bisa yuk 9 jam doang buta,yuk bisa yuk 9 jam doang buta
10850,"Keluaran Resmi Pasaran China , 03 November 2023 Result : 9987 Shio : Kerbau Selamat kepada para pemenang yang !!",keluaran resmi pasaran china 03 november 2023 result 9987 shio kerbau selamat kepada para pemenang yang
2182,cacing militan islam saya baiklihat bahasa uru mengatakan seorang pria besar bhagat terorishinduisme zaman batu maka saya dapat mengatakan cacing ...,cacing militan islam saya baiklihat bahasa uru mengatakan seorang pria besar bhagat terorishinduisme zaman batu maka saya dapat mengatakan cacing ...
29576,"Capres nomor urut 3, Ganjar Pranowo mengajak semua umat beragama agar menghormati perayaan agama lain. Hal itu penting untuk menjaga persatuan dan...",capres nomor urut 3 ganjar pranowo mengajak semua umat beragama agar menghormati perayaan agama lain hal itu penting untuk menjaga persatuan dan k...
17287,"Bagi Gila Prabowo dan Gila Jokowi, politik mungkin adalah bentuk hiburan yang sangat serius.",bagi gila prabowo dan gila jokowi politik mungkin adalah bentuk hiburan yang sangat serius
38045,USER Jawaban yang cerdas. Cebong mana punya otak untuk menakar junjungan nya.\nYa namanya generasi lahir lahir dari got.',user jawaban yang cerdas cebong mana punya otak untuk menakar junjungan nya nya namanya generasi lahir lahir dari got
15158,"Perdana Menteri Israel Benjamin Netanyahu pada Jumat mengatakan bahwa serangan ke Jalur Gaza ""hanyalah permulaan""",perdana menteri israel benjamin netanyahu pada jumat mengatakan bahwa serangan ke jalur gaza hanyalah permulaan
1291,stan punya kata terakhir a sebelum memblokir saya pernah mendengar tentang kaustik sarkasme ubah nama sendiri terima kasih mengerikan mwah xxx,stan punya kata terakhir a sebelum memblokir saya pernah mendengar tentang kaustik sarkasme ubah nama sendiri terima kasih mengerikan mwah xxx
32853,USER Padahal dulu mah smgt kalo masalah ngewe skrg udh males gue mah',user padahal dulu mah smgt kalo masalah ngewe skrg udh males gue mah
18652,"Ini retardretard twitter kalo soal pemilu paling bacot “golput gapapa samasama gabener pilihannya”, tapi kalo soal israel dan *sengaja gamau nyebu...",ini retardretard twitter kalo soal pemilu paling bacot golput gapapa samasama gabener pilihannya tapi kalo soal israel dan sengaja gamau nyebut la...


Average Character Reduction: 24.19 chars/row
Average Word Reduction: 0.74 words/row


In [9]:
# 12. PREPROCESSING QUALITY VALIDATION

empty_processed_mask = (df[TEXT_PROCESSED_COL] == "")
empty_processed_count = empty_processed_mask.sum()

unchanged_mask = (df[TEXT_COL].astype(str) == df[TEXT_PROCESSED_COL])
unchanged_count = unchanged_mask.sum()

print("### Quality Check ###")
print(f"Rows reduced to empty string: {empty_processed_count} ({empty_processed_count/len(df)*100:.2f}%)")
print(f"Rows completely unchanged: {unchanged_count} ({unchanged_count/len(df)*100:.2f}%)")

if empty_processed_count > 0:
    print("\nWARNING: Some texts were completely removed by preprocessing (e.g., they contained only URLs or punctuation).")
    print("These will be dropped to prevent TF-IDF errors.")
    # We drop empty records because TF-IDF cannot process empty strings
    df = df[~empty_processed_mask].copy()
    print(f"Dataset size after dropping empty texts: {len(df)}")


### Quality Check ###
Rows reduced to empty string: 2 (0.00%)
Rows completely unchanged: 3194 (7.44%)

These will be dropped to prevent TF-IDF errors.
Dataset size after dropping empty texts: 42952


In [10]:
# 13. TEXT STATISTICS AFTER PROCESSING

print("### Processed Text Statistics ###")
print(f"Total Rows: {len(df)}")
print(f"Mean Char Count: {df['char_count_proc'].mean():.2f}")
print(f"Mean Word Count: {df['word_count_proc'].mean():.2f}")
print(f"Min Word Count: {df['word_count_proc'].min()}")
print(f"Max Word Count: {df['word_count_proc'].max()}")


### Processed Text Statistics ###
Total Rows: 42952
Mean Char Count: 217.62
Mean Word Count: 33.89
Min Word Count: 1
Max Word Count: 980


In [11]:
# 14. PRESERVE LABELS AND METADATA

# Verify target labels
if df[LABEL_COL].isnull().sum() > 0:
    print("WARNING: Label column contains nulls!")
else:
    print("Target labels preserved perfectly.")

# Ensure original text exists
if TEXT_COL not in df.columns:
    print("WARNING: Original text column missing!")
else:
    print("Original text column preserved perfectly.")

# Drop temporary calculation columns
cols_to_drop = ['char_count_orig', 'word_count_orig', 'char_count_proc', 'word_count_proc', 'char_reduction', 'word_reduction']
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])


Target labels preserved perfectly.
Original text column preserved perfectly.


In [12]:
# 15. SAVE PROCESSED DATASET

# Reorder columns for readability: Original text, Processed Text, Labels, Metadata
ordered_cols = [TEXT_COL, TEXT_PROCESSED_COL, LABEL_COL] + [c for c in df.columns if c not in [TEXT_COL, TEXT_PROCESSED_COL, LABEL_COL]]

processed_df = df[ordered_cols]
processed_df.to_csv(PROCESSED_DATA_PATH, index=False)

print(f"SUCCESS: Processed dataset saved to {PROCESSED_DATA_PATH}")
print(f"Final output shape: {processed_df.shape}")


SUCCESS: Processed dataset saved to /home/zapp/Kampus/PM-NEW/data/processed/processed_cyberbullying_dataset.csv
Final output shape: (42952, 9)


In [13]:
# 16. SAVE PREPROCESSING METADATA

metadata = {
    "input_path": str(VALIDATED_DATA_PATH),
    "output_path": str(PROCESSED_DATA_PATH),
    "initial_rows": int(df.shape[0]) + int(empty_processed_count if 'empty_processed_count' in locals() else 0),
    "final_rows": int(df.shape[0]),
    "dropped_empty_rows": int(empty_processed_count if 'empty_processed_count' in locals() else 0),
    "applied_steps": [
        "Lowercasing",
        "URL Removal",
        "HTML Removal",
        "Username Replacement (USER_MENTION)",
        "Hashtag Strip (Keep word)",
        "Punctuation Removal",
        "Whitespace Normalization"
    ],
    "skipped_steps": [
        "Stopword Removal (Skipped to preserve negation)",
        "Stemming (Skipped to preserve slang)",
        "Repeated Character Normalization (Skipped to preserve emotion)"
    ]
}

# We save as a dataframe just to utilize CSV, or simply write JSON. We'll do JSON.
import json
meta_path = REPORTS_DIR / "preprocessing_config.json"
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=4)

print(f"Preprocessing metadata saved to {meta_path}")


Preprocessing metadata saved to /home/zapp/Kampus/PM-NEW/reports/preprocessing_config.json


# 17. PREPROCESSING SUMMARY

### Input Dataset
- **Path**: `data/validated/validated_cyberbullying_dataset.csv`
- **Output Path**: `data/processed/processed_cyberbullying_dataset.csv`

### Steps Applied
1. Lowercasing
2. URL / HTML Tag Removal
3. Username Mention Replacement
4. Hashtag Punctuation Stripping
5. Non-Alphanumeric Punctuation Removal
6. Whitespace Normalization

### Steps Intentionally Skipped
- **Stopword Removal**: Negations are vital for cyberbullying text.
- **Stemming**: Sastrawi harms informal slang.
- **Repeated Char Normalization**: Preserved for severity signaling.

### Next Step
The text is now clean, space-separated, and ready for vectorization. We will use this output in `notebooks/06_tfidf.ipynb`.


# 18. HOW TO RUN THIS NOTEBOOK

1. Activate the project's virtual environment.
2. Ensure the validated dataset exists in: `data/validated/validated_cyberbullying_dataset.csv`
3. Ensure the validation stage has been completed: `notebooks/04_validation.ipynb`
4. Open: `notebooks/05_preprocessing.ipynb`
5. Verify the configured input dataset path in Step 4.
6. Review the preprocessing strategy in Step 8 to understand why certain standard NLP steps were intentionally skipped.
7. Run the notebook from the first cell to the last cell.
8. Review the before-and-after text comparison in Step 11.
9. Check if any rows were dropped due to becoming completely empty (Step 12).
10. Confirm the processed dataset exists in: `data/processed/processed_cyberbullying_dataset.csv`
11. Proceed to: `notebooks/06_tfidf.ipynb`
